# Teaching an LLM to Count
## Udacity Generative AI Nanodegree - Fundamentals Project

This notebook teaches a small language model (**Qwen/Qwen2.5-3B-Instruct**) to accurately count letter occurrences in words using **GRPO (Group Relative Policy Optimization)** — a reinforcement learning technique.

### Phases
1. **Model Setup** — Load model with LoRA adapters
2. **Baseline Prompting** — Chain-of-thought system prompt with few-shot examples
3. **Dataset Creation** — Generate letter-counting training samples
4. **Reward Functions** — Five custom reward signals for GRPO
5. **Training** — GRPO training with reward curves
6. **Results** — Before/after comparison and catastrophic forgetting check

## Phase 0: Install Dependencies

In [ ]:
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes matplotlib

## Phase 1: Model Setup

Load `Qwen/Qwen2.5-3B-Instruct` with a LoRA configuration targeting all attention and MLP projection layers. We use 4-bit quantization to fit within GPU memory constraints.

In [ ]:
import torch
import re
import random
import matplotlib.pyplot as plt
from collections import defaultdict

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# Model name
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=64,  # lora_rank
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Phase 2: Baseline Prompting

We define a system prompt that teaches the model a structured counting approach:
1. Number each letter in the word sequentially
2. Spell out each letter and track a running count of the target letter
3. Output the final answer in a structured format

The few-shot example demonstrates the expected reasoning pattern.

In [ ]:
SYSTEM_PROMPT = """You are a letter-counting assistant. When asked how many times a letter appears in a word, you must:

1. Number each letter in the word sequentially (1. letter, 2. letter, ...)
2. For each letter, state what it is and keep a running count of the target letter
3. Give the final answer

You MUST use this exact format:
<reasoning>
1. [letter]: [is/is not target] (count so far: N)
2. [letter]: [is/is not target] (count so far: N)
...
</reasoning>
<answer>[number]</answer>

Example:
Question: How many times does the letter 'o' appear in the word 'room'?

<reasoning>
1. r: not target (count so far: 0)
2. o: is target (count so far: 1)
3. o: is target (count so far: 2)
4. m: not target (count so far: 2)
</reasoning>
<answer>2</answer>

Always go through EVERY letter one by one. Never skip letters or guess."""

In [ ]:
# Test baseline model (before training)
def generate_response(model, tokenizer, prompt, system_prompt=SYSTEM_PROMPT, max_new_tokens=512):
    """Generate a response from the model given a prompt."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Baseline test
test_prompt = "How many times does the letter 'e' appear in the word 'elephant'?"
print("=== BASELINE (Before Training) ===")
print(f"Question: {test_prompt}")
print(f"Expected answer: 2")
print()
baseline_response = generate_response(model, tokenizer, test_prompt)
print(f"Model response:\n{baseline_response}")

## Phase 3: Dataset Creation

We generate training samples by pairing common English words with target letters. Each record contains:
- A formatted prompt asking the model to count a specific letter in a word
- The ground truth answer (actual count)
- Metadata for reward computation (the word and target letter)

In [ ]:
# Word list (~50 common English words with varied letter patterns)
ALL_WORDS = [
    "apple", "banana", "orange", "strawberry", "elephant",
    "giraffe", "butterfly", "computer", "telephone", "beautiful",
    "chocolate", "mountain", "umbrella", "pineapple", "watermelon",
    "dictionary", "adventure", "happiness", "wonderful", "education",
    "environment", "strawberry", "necessary", "opportunity", "communication",
    "international", "professional", "understanding", "responsibility", "extraordinary",
    "accommodation", "appreciation", "recommendation", "mississippi", "bookkeeper",
    "committee", "millennium", "broccoli", "cappuccino", "occurrence",
    "possession", "accessible", "successfully", "assassination", "balloon",
    "beginning", "different", "disappear", "embarrass", "exaggerate",
]

# Target letters to practice
TARGET_LETTERS = list("abcdefghilmnoprstuw")


def generate_records(words, target_letters):
    """Generate training records for letter-counting tasks.
    
    Each record contains:
    - prompt: The chat-formatted prompt for the model
    - answer: The ground truth count (as string)
    - word: The original word
    - target_letter: The letter being counted
    """
    records = []
    
    for word in words:
        for letter in target_letters:
            count = word.lower().count(letter)
            # Include both zero and non-zero counts for balanced training
            # but favor non-zero cases (more informative)
            if count == 0 and random.random() > 0.3:
                continue
            
            prompt = f"How many times does the letter '{letter}' appear in the word '{word}'?"
            
            # Format as chat messages
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ]
            formatted_prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            
            records.append({
                "prompt": formatted_prompt,
                "answer": str(count),
                "word": word,
                "target_letter": letter,
            })
    
    random.seed(42)
    random.shuffle(records)
    return records


# Generate the dataset
records = generate_records(ALL_WORDS, TARGET_LETTERS)
dataset = Dataset.from_list(records)

print(f"Total training samples: {len(dataset)}")
print(f"\nSample record:")
print(f"  Word: {dataset[0]['word']}")
print(f"  Target letter: {dataset[0]['target_letter']}")
print(f"  Answer: {dataset[0]['answer']}")
print(f"  Prompt (last 100 chars): ...{dataset[0]['prompt'][-100:]}")

## Phase 4: Reward Functions

Five reward functions guide the GRPO training. Each evaluates a different aspect of the model's counting behavior:

| Reward | What it checks | Range |
|--------|---------------|-------|
| `numbering_reward_func` | Sequential numbering (1., 2., 3., ...) | [-1.0, +0.5] |
| `spelling_reward_func` | Correct letter-by-letter spelling | [-1.0, +2.0] |
| `counting_reward_func` | Accurate running count | [-1.0, +1.0] |
| `format_reward_func` | Proper `<reasoning>...<answer>` format | [0.0, +1.0] |
| `correct_answer_reward_func` | Final answer correctness | [-1.0, +2.0] |

In [ ]:
def _extract_reasoning_and_answer(text):
    """Helper to extract reasoning and answer sections from model output."""
    reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
    answer_match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""
    answer = answer_match.group(1).strip() if answer_match else ""
    return reasoning, answer


def _parse_numbered_lines(reasoning):
    """Parse numbered lines from reasoning, returning list of (number, content) tuples."""
    pattern = r"(\d+)\.\s*(.*?)(?=\n\d+\.|\Z)"
    matches = re.findall(pattern, reasoning, re.DOTALL)
    return [(int(num), content.strip()) for num, content in matches]


def numbering_reward_func(completions, **kwargs):
    """Check sequential numbering of letters in the reasoning.
    
    Rewards:
    - Correct sequential numbering matching word length: +0.5
    - Wrong numbering (gaps or wrong sequence): -0.5
    - Too many numbers (overflow past word length): -1.0
    """
    words = kwargs.get("word", [])
    rewards = []
    
    for i, completion in enumerate(completions):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        word = words[i] if i < len(words) else ""
        expected_len = len(word)
        
        reasoning, _ = _extract_reasoning_and_answer(content)
        numbered_lines = _parse_numbered_lines(reasoning)
        
        if not numbered_lines:
            rewards.append(-0.5)
            continue
        
        # Check if numbers overflow past word length
        if len(numbered_lines) > expected_len:
            rewards.append(-1.0)
            continue
        
        # Check sequential numbering: 1, 2, 3, ...
        correct = all(
            numbered_lines[j][0] == j + 1
            for j in range(len(numbered_lines))
        )
        
        if correct and len(numbered_lines) == expected_len:
            rewards.append(0.5)
        else:
            rewards.append(-0.5)
    
    return rewards


def spelling_reward_func(completions, **kwargs):
    """Check letter-by-letter spelling accuracy in the reasoning.
    
    Rewards:
    - Perfect spelling (all letters correct, right count): +2.0
    - Penalty for length difference: -0.5 per character difference
    - Extra letters not in word: -1.0 each
    - Missing letters from word: -0.5 each
    """
    words = kwargs.get("word", [])
    rewards = []
    
    for i, completion in enumerate(completions):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        word = words[i].lower() if i < len(words) else ""
        
        reasoning, _ = _extract_reasoning_and_answer(content)
        numbered_lines = _parse_numbered_lines(reasoning)
        
        if not numbered_lines:
            rewards.append(-0.5)
            continue
        
        # Extract the letter mentioned in each numbered line
        spelled_letters = []
        for _, line_content in numbered_lines:
            # Match patterns like "a:" or "a :" at the start of the line content
            letter_match = re.match(r"([a-zA-Z])\s*:", line_content)
            if letter_match:
                spelled_letters.append(letter_match.group(1).lower())
        
        # Compare spelled letters with actual word letters
        word_letters = list(word)
        
        if spelled_letters == word_letters:
            rewards.append(2.0)
            continue
        
        reward = 0.0
        
        # Penalty for length difference
        len_diff = abs(len(spelled_letters) - len(word_letters))
        reward -= 0.5 * len_diff
        
        # Check for extra letters (not matching expected position)
        for j, letter in enumerate(spelled_letters):
            if j < len(word_letters):
                if letter != word_letters[j]:
                    reward -= 1.0  # Wrong letter at this position
            else:
                reward -= 1.0  # Extra letter beyond word length
        
        # Missing letters
        if len(spelled_letters) < len(word_letters):
            reward -= 0.5 * (len(word_letters) - len(spelled_letters))
        
        rewards.append(max(reward, -1.0))  # Floor at -1.0
    
    return rewards


def counting_reward_func(completions, **kwargs):
    """Check running count accuracy at each step.
    
    For each numbered line, verify the running count is correct.
    Rewards:
    - Correct step: +1.0 (normalized by word length)
    - Incorrect step: -1.0 (normalized by word length)
    """
    words = kwargs.get("word", [])
    target_letters = kwargs.get("target_letter", [])
    rewards = []
    
    for i, completion in enumerate(completions):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        word = words[i].lower() if i < len(words) else ""
        target = target_letters[i].lower() if i < len(target_letters) else ""
        
        reasoning, _ = _extract_reasoning_and_answer(content)
        numbered_lines = _parse_numbered_lines(reasoning)
        
        if not numbered_lines:
            rewards.append(-1.0)
            continue
        
        # Compute expected running counts
        expected_counts = []
        running = 0
        for ch in word:
            if ch == target:
                running += 1
            expected_counts.append(running)
        
        # Extract actual running counts from each line
        total_score = 0.0
        num_steps = min(len(numbered_lines), len(word))
        
        for j in range(num_steps):
            _, line_content = numbered_lines[j]
            # Look for "count so far: N" pattern
            count_match = re.search(r"count\s+so\s+far:\s*(\d+)", line_content)
            if count_match:
                actual_count = int(count_match.group(1))
                if actual_count == expected_counts[j]:
                    total_score += 1.0
                else:
                    total_score -= 1.0
            else:
                total_score -= 1.0  # Missing count
        
        # Normalize by word length
        if len(word) > 0:
            normalized = total_score / len(word)
        else:
            normalized = 0.0
        
        rewards.append(max(min(normalized, 1.0), -1.0))
    
    return rewards


def format_reward_func(completions, **kwargs):
    """Check output format compliance.
    
    Rewards:
    - Has <reasoning>...</reasoning><answer>...</answer>: +0.5
    - Answer is numeric: +0.5
    """
    rewards = []
    
    for completion in completions:
        content = completion[0]["content"] if isinstance(completion, list) else completion
        reward = 0.0
        
        # Check for reasoning and answer tags
        has_reasoning = bool(re.search(r"<reasoning>.*?</reasoning>", content, re.DOTALL))
        has_answer = bool(re.search(r"<answer>.*?</answer>", content, re.DOTALL))
        
        if has_reasoning and has_answer:
            reward += 0.5
        
        # Check if answer is numeric
        answer_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if answer_match:
            answer_text = answer_match.group(1).strip()
            if answer_text.isdigit():
                reward += 0.5
        
        rewards.append(reward)
    
    return rewards


def correct_answer_reward_func(completions, **kwargs):
    """Check if the final answer is correct.
    
    Rewards:
    - Correct answer: +2.0
    - Incorrect answer: -1.0
    """
    answers = kwargs.get("answer", [])
    rewards = []
    
    for i, completion in enumerate(completions):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        expected = answers[i] if i < len(answers) else ""
        
        _, extracted_answer = _extract_reasoning_and_answer(content)
        
        if extracted_answer == expected:
            rewards.append(2.0)
        else:
            rewards.append(-1.0)
    
    return rewards


print("All 5 reward functions defined.")
print("Testing on a sample completion...")

# Quick sanity test
sample_completion = [[{"content": """<reasoning>
1. r: not target (count so far: 0)
2. o: is target (count so far: 1)
3. o: is target (count so far: 2)
4. m: not target (count so far: 2)
</reasoning>
<answer>2</answer>"""}]]

test_kwargs = {
    "word": ["room"],
    "target_letter": ["o"],
    "answer": ["2"],
}

print(f"  numbering_reward:      {numbering_reward_func(sample_completion, **test_kwargs)}")
print(f"  spelling_reward:       {spelling_reward_func(sample_completion, **test_kwargs)}")
print(f"  counting_reward:       {counting_reward_func(sample_completion, **test_kwargs)}")
print(f"  format_reward:         {format_reward_func(sample_completion, **test_kwargs)}")
print(f"  correct_answer_reward: {correct_answer_reward_func(sample_completion, **test_kwargs)}")

## Phase 5: GRPO Training

We use **GRPOTrainer** from the `trl` library to train the model with our five reward functions. GRPO generates multiple completions per prompt, scores them with the reward functions, and updates the policy to favor higher-reward outputs.

### Training Strategy
- **Quick run (5 steps)**: Verify everything works end-to-end
- **Full run (80-100 steps)**: Actual training with reward curve visualization

In [ ]:
# ============================================================
# Quick verification run (5 steps) — change max_steps for full training
# ============================================================

# Set max_steps: use 5 for quick verification, 80-100 for full training
MAX_STEPS = 100  # <-- Change to 5 for quick test, 100 for full training

training_args = GRPOConfig(
    output_dir="./results",
    learning_rate=10e-6,
    beta=0.0001,
    per_device_train_batch_size=16,
    num_generations=4,
    gradient_accumulation_steps=1,
    max_steps=MAX_STEPS,
    max_completion_length=512,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=25,
    report_to="none",  # Disable wandb/tensorboard
    bf16=True,
    remove_unused_columns=False,
)

# Initialize GRPO Trainer with all 5 reward functions
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        numbering_reward_func,
        spelling_reward_func,
        counting_reward_func,
        format_reward_func,
        correct_answer_reward_func,
    ],
    processing_class=tokenizer,
)

print(f"Trainer initialized. Starting training for {MAX_STEPS} steps...")

In [ ]:
# Run training
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Final loss: {train_result.training_loss:.4f}")

### Reward Curves

Plot the reward progression across training steps to visualize learning dynamics.

In [ ]:
# Extract and plot reward curves from training logs
log_history = trainer.state.log_history

# Collect metrics
steps = []
reward_keys = {
    "reward/numbering_reward_func": "Numbering",
    "reward/spelling_reward_func": "Spelling",
    "reward/counting_reward_func": "Counting",
    "reward/format_reward_func": "Format",
    "reward/correct_answer_reward_func": "Correct Answer",
}

metrics_data = defaultdict(list)
step_data = defaultdict(list)

for entry in log_history:
    if "step" not in entry:
        continue
    step = entry["step"]
    for key, label in reward_keys.items():
        if key in entry:
            metrics_data[label].append(entry[key])
            step_data[label].append(step)

# Also collect total reward and loss
total_reward_steps = []
total_rewards = []
loss_steps = []
losses = []

for entry in log_history:
    if "step" not in entry:
        continue
    if "reward" in entry:
        total_reward_steps.append(entry["step"])
        total_rewards.append(entry["reward"])
    if "loss" in entry:
        loss_steps.append(entry["step"])
        losses.append(entry["loss"])

# Plot individual reward curves
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("GRPO Training - Reward Curves", fontsize=16)

colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
for idx, (label, values) in enumerate(metrics_data.items()):
    ax = axes[idx // 3][idx % 3]
    ax.plot(step_data[label], values, color=colors[idx], alpha=0.7, linewidth=1.5)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)

# Plot total reward in the last subplot
ax = axes[1][2]
if total_rewards:
    ax.plot(total_reward_steps, total_rewards, color="#795548", alpha=0.7, linewidth=1.5)
ax.set_title("Total Reward", fontsize=12)
ax.set_xlabel("Step")
ax.set_ylabel("Reward")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reward_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot loss curve
if losses:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(loss_steps, losses, color="#E91E63", alpha=0.7, linewidth=1.5)
    ax.set_title("Training Loss", fontsize=14)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Plots saved to reward_curves.png and loss_curve.png")

## Phase 6: Results

### Save the LoRA Adapter
Save the fine-tuned LoRA weights for later use.

In [ ]:
# Save the LoRA adapter
ADAPTER_PATH = "./letter_counting_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

### Compare Old and New Model

Side-by-side comparison of the base model vs. the GRPO-trained model on several letter-counting tasks.

In [ ]:
def compare_old_and_new_model():
    """Compare the base model and the GRPO-trained model on letter-counting tasks."""
    
    # Load the base model (without LoRA) for comparison
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    
    # Test cases: (word, letter, expected_count)
    test_cases = [
        ("strawberry", "r", 3),
        ("mississippi", "s", 4),
        ("banana", "a", 3),
        ("bookkeeper", "e", 2),
        ("communication", "m", 2),
        ("butterfly", "t", 2),
        ("chocolate", "c", 2),
        ("exaggerate", "g", 2),
    ]
    
    print("=" * 80)
    print("MODEL COMPARISON: Base vs. GRPO-Trained")
    print("=" * 80)
    
    base_correct = 0
    trained_correct = 0
    
    for word, letter, expected in test_cases:
        prompt = f"How many times does the letter '{letter}' appear in the word '{word}'?"
        
        # Base model response
        base_response = generate_response(base_model, tokenizer, prompt)
        _, base_answer = _extract_reasoning_and_answer(base_response)
        
        # Trained model response
        trained_response = generate_response(model, tokenizer, prompt)
        _, trained_answer = _extract_reasoning_and_answer(trained_response)
        
        base_ok = base_answer == str(expected)
        trained_ok = trained_answer == str(expected)
        
        if base_ok:
            base_correct += 1
        if trained_ok:
            trained_correct += 1
        
        print(f"\n{'─' * 60}")
        print(f"Q: '{letter}' in '{word}'?  Expected: {expected}")
        print(f"  Base model answer:    {base_answer:>3s}  {'✓' if base_ok else '✗'}")
        print(f"  Trained model answer: {trained_answer:>3s}  {'✓' if trained_ok else '✗'}")
    
    print(f"\n{'=' * 60}")
    print(f"ACCURACY SUMMARY")
    print(f"  Base model:    {base_correct}/{len(test_cases)} ({100*base_correct/len(test_cases):.0f}%)")
    print(f"  Trained model: {trained_correct}/{len(test_cases)} ({100*trained_correct/len(test_cases):.0f}%)")
    print(f"{'=' * 60}")
    
    # Clean up base model
    del base_model
    torch.cuda.empty_cache()

compare_old_and_new_model()

### Catastrophic Forgetting Check

Verify the model still handles general knowledge questions after fine-tuning. If the model can only count letters but forgets everything else, the training has caused catastrophic forgetting.

In [ ]:
# Test for catastrophic forgetting with general knowledge questions
general_questions = [
    "What is the capital of France?",
    "Explain photosynthesis in one sentence.",
    "What is 15 * 23?",
]

general_system_prompt = "You are a helpful assistant. Answer questions concisely."

print("=" * 60)
print("CATASTROPHIC FORGETTING CHECK")
print("=" * 60)

for question in general_questions:
    response = generate_response(
        model, tokenizer, question,
        system_prompt=general_system_prompt,
        max_new_tokens=150,
    )
    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 40)

print("\nIf the answers above are coherent and relevant, the model has NOT")
print("suffered from catastrophic forgetting. LoRA fine-tuning preserves")
print("general knowledge while adding the letter-counting capability.")

### Detailed Trained Model Output

Show the full reasoning chain from the trained model to demonstrate the structured counting behavior learned through GRPO.

In [ ]:
# Show full detailed outputs from the trained model
detailed_tests = [
    ("How many times does the letter 'r' appear in the word 'strawberry'?", 3),
    ("How many times does the letter 's' appear in the word 'mississippi'?", 4),
    ("How many times does the letter 'l' appear in the word 'successfully'?", 2),
]

print("=" * 60)
print("DETAILED TRAINED MODEL OUTPUTS")
print("=" * 60)

for prompt, expected in detailed_tests:
    response = generate_response(model, tokenizer, prompt)
    _, answer = _extract_reasoning_and_answer(response)
    correct = answer == str(expected)
    
    print(f"\n{'─' * 60}")
    print(f"PROMPT: {prompt}")
    print(f"EXPECTED: {expected}")
    print(f"MODEL OUTPUT:")
    print(response)
    print(f"\nVERDICT: {'CORRECT' if correct else 'INCORRECT'} (extracted answer: {answer})")

print(f"\n{'=' * 60}")
print("Project complete!")